Dataset cleanup

In [2]:
import pandas as pd
import os

# Keep only columns from RNG_SEED onward
def trim(path):
    df = pd.read_csv(path)
    return df.loc[:, df.columns[df.columns.get_loc('RNG_SEED'):]]

a = trim('original_csv_files/full_sweep.csv')
b = trim('original_csv_files/random_sweep.csv')

# Ensure the remaining column headers match
assert list(a.columns) == list(b.columns), 'Column headers do not match!'

# Combine and shuffle rows
full = pd.concat([a, b], ignore_index=True)
full = full.sample(frac=1, random_state=42).reset_index(drop=True)

# Drop columns we don't model on:
#   RNG_SEED         - simulation random seed, a pure noise input
#   citric_total_mmol - ~zero variance (all values ~0), nothing to predict
full = full.drop(columns=['RNG_SEED', 'citric_total_mmol'])

# Drop any rows that have blank cells
full = full.dropna().reset_index(drop=True)

# Save combined set
os.makedirs('model_csv_files', exist_ok=True)
full.to_csv('model_csv_files/full_set.csv', index=False)

# 80/20 split into training and validation sets
n_train = int(len(full) * 0.8)
full.iloc[:n_train].to_csv('model_csv_files/training_set.csv', index=False)
full.iloc[n_train:].to_csv('model_csv_files/validation_set.csv', index=False)

print(f'Columns: {len(full.columns)} | Total: {len(full)} | Train: {n_train} | Val: {len(full) - n_train}')

Columns: 17 | Total: 2812 | Train: 2249 | Val: 563


Surrogate model training and validation

Linear

In [3]:
import pandas as pd
from sklearn import linear_model
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

scale = StandardScaler()

train_set = pd.read_csv("model_csv_files/training_set.csv")
valid_set = pd.read_csv("model_csv_files/validation_set.csv")

feature_cols = ['MU_MAX','DEPLETION_RADIUS','HYPHA_RADIUS_UM','PSI_BRANCH','MIN_BRANCH_ANGLE_DEG','APICAL_BRANCH_MIN_STEP','LATERAL_DISTANCE_FROM_TIP','PHI_N','PHI_P','LATERAL_BRANCH_MIN_STEP','MAX_DEVIATION_DEG','nutrient_pm_fraction']

# Fit the scaler on the training data only, then apply it to both sets
train_x = scale.fit_transform(train_set[feature_cols])
valid_x = scale.transform(valid_set[feature_cols])

y1 = train_set['avg_length_um']
y2 = train_set['avg_volume_um3']
y3 = train_set['total_tips']
y4 = train_set['n_apical_events']
y5 = train_set['n_lateral_events']

yv1 = valid_set['avg_length_um']
yv2 = valid_set['avg_volume_um3']
yv3 = valid_set['total_tips']
yv4 = valid_set['n_apical_events']
yv5 = valid_set['n_lateral_events']

y_list = [y1,y2, y3, y4, y5]
yv_list = [yv1, yv2, yv3, yv4, yv5]

regr = linear_model.LinearRegression()

for y, yv in zip(y_list, yv_list):
    print(f'Results for {y.name}:')
    regr.fit(train_x,y)
    print(f'Training coefficients: \n{regr.coef_}')
    print(f'Training r2 score: {r2_score(y, regr.predict(train_x))}')
    print(f'Validation r2 score: {r2_score(yv, regr.predict(valid_x))}\n')


Results for avg_length_um:
Training coefficients: 
[ 216.10069081   90.39658225   -4.6569784   153.23927686  -69.40205378
 -143.61405567  -28.25495446   -5.37859611   15.25927379 -375.97225891
   36.6892904    43.60724858]
Training r2 score: 0.34049870694390905
Validation r2 score: 0.24918850865882014

Results for avg_volume_um3:
Training coefficients: 
[ 1691.54614932   707.57431033  3210.24609794  1540.45470752
 -1029.09120823 -1381.57609583  -386.5441515    -96.93617947
   177.70708588 -3398.70219997    13.77077174   573.79152463]
Training r2 score: 0.25448157994980525
Validation r2 score: 0.16902025813956245

Results for total_tips:
Training coefficients: 
[  20.47863501   29.78777286    7.85933447   48.08226189  -23.64937155
  -40.16480794  -20.2849871     0.24923243    5.20946045 -101.43432338
    1.08650654   11.84618305]
Training r2 score: 0.2903847883209828
Validation r2 score: 0.24286212404434715

Results for n_apical_events:
Training coefficients: 
[  4.63302106  13.00721869

In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

train_set = pd.read_csv("model_csv_files/training_set.csv")
valid_set = pd.read_csv("model_csv_files/validation_set.csv")

feature_cols = ['MU_MAX','DEPLETION_RADIUS','HYPHA_RADIUS_UM','PSI_BRANCH','MIN_BRANCH_ANGLE_DEG','APICAL_BRANCH_MIN_STEP','LATERAL_DISTANCE_FROM_TIP','PHI_N','PHI_P','LATERAL_BRANCH_MIN_STEP','MAX_DEVIATION_DEG','nutrient_pm_fraction']
target_cols = ['avg_length_um','avg_volume_um3','total_tips','n_apical_events','n_lateral_events']

# Scale features on the training set only, then apply to both
x_scale = StandardScaler()
train_x = x_scale.fit_transform(train_set[feature_cols])
valid_x = x_scale.transform(valid_set[feature_cols])

results = []
for target in target_cols:
    y_train = train_set[target]
    y_valid = valid_set[target]

    # SVR is sensitive to target scale, so standardize y as well
    y_scale = StandardScaler()
    y_train_s = y_scale.fit_transform(y_train.values.reshape(-1, 1)).ravel()

    svr = SVR(kernel='rbf', C=100, epsilon=0.5, gamma=0.08)
    svr.fit(train_x, y_train_s)

    # Predict, then invert the target scaling back to original units
    train_pred = y_scale.inverse_transform(svr.predict(train_x).reshape(-1, 1)).ravel()
    pred = y_scale.inverse_transform(svr.predict(valid_x).reshape(-1, 1)).ravel()

    results.append({
        'target': target,
        'train_r2': r2_score(y_train, train_pred),
        'valid_r2': r2_score(y_valid, pred),
        'MAE': mean_absolute_error(y_valid, pred),
        'RMSE': np.sqrt(mean_squared_error(y_valid, pred)),
        'y_mean': y_valid.mean(),
    })

results_df = pd.DataFrame(results).set_index('target')
# RMSE relative to the mean of the target — a scale-free sense of error size
results_df['RMSE_%_of_mean'] = 100 * results_df['RMSE'] / results_df['y_mean']
print(results_df.round(4).to_string())


                  train_r2  valid_r2        MAE       RMSE     y_mean  RMSE_%_of_mean
target                                                                               
avg_length_um       0.9110    0.5022   297.1353   558.8942   331.6489        168.5199
avg_volume_um3      0.8867    0.3819  4427.6850  7089.6573  2453.2636        288.9888
total_tips          0.9138    0.4652    84.6885   156.2491    81.7880        191.0416
n_apical_events     0.8888    0.2838    53.6405    84.2539    31.9702        263.5392
n_lateral_events    0.9137    0.5774    50.6093    86.8437    45.8933        189.2294


In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

train_set = pd.read_csv("model_csv_files/training_set.csv")
valid_set = pd.read_csv("model_csv_files/validation_set.csv")

feature_cols = ['MU_MAX','DEPLETION_RADIUS','HYPHA_RADIUS_UM','PSI_BRANCH','MIN_BRANCH_ANGLE_DEG','APICAL_BRANCH_MIN_STEP','LATERAL_DISTANCE_FROM_TIP','PHI_N','PHI_P','LATERAL_BRANCH_MIN_STEP','MAX_DEVIATION_DEG','nutrient_pm_fraction']
target_cols = ['avg_length_um','avg_volume_um3','total_tips','n_apical_events','n_lateral_events']

train_x = train_set[feature_cols]
valid_x = valid_set[feature_cols]

# Feature pipeline: standardize X, then SVR.
base = Pipeline([('scaler', StandardScaler()), ('svr', SVR(kernel='rbf'))])
# Target pipeline: log1p (tame the 6-9 skew) then standardize. Both are
# re-fit inside every CV fold, and predictions are auto-inverted to raw units.
y_transform = Pipeline([
    ('log', FunctionTransformer(np.log1p, inverse_func=np.expm1)),
    ('scale', StandardScaler()),
])
model = TransformedTargetRegressor(regressor=base, transformer=y_transform)

param_grid = {
    'regressor__svr__C': [1, 10, 100, 300, 1000],
    'regressor__svr__epsilon': [0.01, 0.05, 0.1, 0.5],
    'regressor__svr__gamma': ['scale', 0.01, 0.05, 0.1, 0.5],
}

results = []
for target in target_cols:
    y_train = train_set[target]
    y_valid = valid_set[target]

    search = GridSearchCV(model, param_grid, scoring='r2', cv=5, n_jobs=-1)
    search.fit(train_x, y_train)

    train_pred = search.predict(train_x)                 # raw units
    pred = search.predict(valid_x)                       # raw units
    # R2 in log space is the fair metric when the target spans orders of magnitude.
    best = search.best_params_
    results.append({
        'target': target,
        'cv_r2': search.best_score_,
        'train_r2': r2_score(y_train, train_pred),
        'valid_r2': r2_score(y_valid, pred),
        'train_r2_log': r2_score(np.log1p(y_train), np.log1p(np.clip(train_pred, 0, None))),
        'valid_r2_log': r2_score(np.log1p(y_valid), np.log1p(np.clip(pred, 0, None))),
        'MAE': mean_absolute_error(y_valid, pred),
        'RMSE': np.sqrt(mean_squared_error(y_valid, pred)),
        'C': best['regressor__svr__C'],
        'epsilon': best['regressor__svr__epsilon'],
        'gamma': best['regressor__svr__gamma'],
    })
    print(f"{target}: CV r2={search.best_score_:.3f}  params={best}")

results_df = pd.DataFrame(results).set_index('target')
print('\n' + results_df.round(4).to_string())


avg_length_um: CV r2=0.506  params={'regressor__svr__C': 1, 'regressor__svr__epsilon': 0.1, 'regressor__svr__gamma': 0.05}
avg_volume_um3: CV r2=0.499  params={'regressor__svr__C': 1, 'regressor__svr__epsilon': 0.01, 'regressor__svr__gamma': 0.05}
total_tips: CV r2=0.562  params={'regressor__svr__C': 1, 'regressor__svr__epsilon': 0.1, 'regressor__svr__gamma': 0.05}
n_apical_events: CV r2=0.134  params={'regressor__svr__C': 1, 'regressor__svr__epsilon': 0.5, 'regressor__svr__gamma': 0.05}
n_lateral_events: CV r2=0.421  params={'regressor__svr__C': 10, 'regressor__svr__epsilon': 0.5, 'regressor__svr__gamma': 0.01}

                   cv_r2  train_r2  valid_r2  train_r2_log  valid_r2_log        MAE       RMSE   C  epsilon  gamma
target                                                                                                            
avg_length_um     0.5057    0.7273    0.7268        0.7511        0.6424   149.0586   414.0684   1     0.10   0.05
avg_volume_um3    0.4989    0.7594

Gradient boost

In [6]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

train_set = pd.read_csv("model_csv_files/training_set.csv")
valid_set = pd.read_csv("model_csv_files/validation_set.csv")

feature_cols = ['MU_MAX','DEPLETION_RADIUS','HYPHA_RADIUS_UM','PSI_BRANCH','MIN_BRANCH_ANGLE_DEG','APICAL_BRANCH_MIN_STEP','LATERAL_DISTANCE_FROM_TIP','PHI_N','PHI_P','LATERAL_BRANCH_MIN_STEP','MAX_DEVIATION_DEG','nutrient_pm_fraction']
target_cols = ['avg_length_um','avg_volume_um3','total_tips','n_apical_events','n_lateral_events']

train_x = train_set[feature_cols]
valid_x = valid_set[feature_cols]

# Gradient boosting on a log1p-transformed target (tames the 6-9 skew).
# Trees need no feature scaling, so no StandardScaler here.
results = []
for target in target_cols:
    y_train = train_set[target]
    y_valid = valid_set[target]

    model = TransformedTargetRegressor(
        regressor=HistGradientBoostingRegressor(
            max_iter=800, learning_rate=0.3, max_leaf_nodes=31, random_state=47,
        ),
        func=np.log1p, inverse_func=np.expm1,
    )
    model.fit(train_x, y_train)
    train_pred = model.predict(train_x)
    pred = model.predict(valid_x)

    results.append({
        'target': target,
        'train_r2': r2_score(y_train, train_pred),
        'valid_r2': r2_score(y_valid, pred),
        'train_r2_log': r2_score(np.log1p(y_train), np.log1p(np.clip(train_pred, 0, None))),
        'valid_r2_log': r2_score(np.log1p(y_valid), np.log1p(np.clip(pred, 0, None))),
        'MAE': mean_absolute_error(y_valid, pred),
        'RMSE': np.sqrt(mean_squared_error(y_valid, pred)),
    })

results_df = pd.DataFrame(results).set_index('target')
print(results_df.round(4).to_string())


                  train_r2  valid_r2  train_r2_log  valid_r2_log       MAE       RMSE
target                                                                               
avg_length_um       0.9999    0.8274        0.9985        0.9517   88.9509   329.1351
avg_volume_um3      1.0000    0.9181        0.9992        0.9491  713.5942  2581.0408
total_tips          0.9999    0.8726        0.9860        0.9183   22.3037    76.2567
n_apical_events     0.9997    0.7995        0.9687        0.8736   12.2425    44.5785
n_lateral_events    1.0000    0.8507        1.0000        0.9666   13.6732    51.6227


Random forest

In [7]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

train_set = pd.read_csv("model_csv_files/training_set.csv")
valid_set = pd.read_csv("model_csv_files/validation_set.csv")

feature_cols = ['MU_MAX','DEPLETION_RADIUS','HYPHA_RADIUS_UM','PSI_BRANCH','MIN_BRANCH_ANGLE_DEG','APICAL_BRANCH_MIN_STEP','LATERAL_DISTANCE_FROM_TIP','PHI_N','PHI_P','LATERAL_BRANCH_MIN_STEP','MAX_DEVIATION_DEG','nutrient_pm_fraction']
target_cols = ['avg_length_um','avg_volume_um3','total_tips','n_apical_events','n_lateral_events']

train_x, valid_x = train_set[feature_cols], valid_set[feature_cols]

# Random Forest is natively multi-output: one model predicts all 5 targets.
# Trees need no scaling; RF is robust to the skew so no log transform needed.
rf = RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42)
rf.fit(train_x, train_set[target_cols])
train_pred = rf.predict(train_x)
pred = rf.predict(valid_x)

results = []
for i, target in enumerate(target_cols):
    y_train = train_set[target]
    y_valid = valid_set[target]
    tp, p = train_pred[:, i], pred[:, i]
    results.append({
        'target': target,
        'train_r2': r2_score(y_train, tp),
        'valid_r2': r2_score(y_valid, p),
        'train_r2_log': r2_score(np.log1p(y_train), np.log1p(np.clip(tp, 0, None))),
        'valid_r2_log': r2_score(np.log1p(y_valid), np.log1p(np.clip(p, 0, None))),
        'MAE': mean_absolute_error(y_valid, p),
        'RMSE': np.sqrt(mean_squared_error(y_valid, p)),
    })

results_df = pd.DataFrame(results).set_index('target')
print(results_df.round(4).to_string())


                  train_r2  valid_r2  train_r2_log  valid_r2_log        MAE       RMSE
target                                                                                
avg_length_um       0.9486    0.4857        0.5123        0.0492   218.9146   568.0661
avg_volume_um3      0.9614    0.4860        0.5421        0.1516  1738.3795  6465.4280
total_tips          0.9394    0.3414        0.7675        0.3266    65.3465   173.3950
n_apical_events     0.9235    0.1442        0.3176       -0.4730    37.5585    92.0994
n_lateral_events    0.9461    0.4931        0.7282        0.3576    35.4811    95.1076


Gaussian Process Regressor

In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

train_set = pd.read_csv("model_csv_files/training_set.csv")
valid_set = pd.read_csv("model_csv_files/validation_set.csv")

feature_cols = ['MU_MAX','DEPLETION_RADIUS','HYPHA_RADIUS_UM','PSI_BRANCH','MIN_BRANCH_ANGLE_DEG','APICAL_BRANCH_MIN_STEP','LATERAL_DISTANCE_FROM_TIP','PHI_N','PHI_P','LATERAL_BRANCH_MIN_STEP','MAX_DEVIATION_DEG','nutrient_pm_fraction']
target_cols = ['avg_length_um','avg_volume_um3','total_tips','n_apical_events','n_lateral_events']

train_x, valid_x = train_set[feature_cols], valid_set[feature_cols]

# ARD kernel: one length-scale per feature + a WhiteKernel to absorb the
# per-seed stochastic noise. GP needs scaled features and (given the skew) a
# log-transformed target.
kernel = (ConstantKernel(1.0) * RBF(length_scale=[1.0] * len(feature_cols))
          + WhiteKernel(noise_level=1.0))

results = []
for target in target_cols:
    y_train = train_set[target]
    y_valid = valid_set[target]

    # normalize_y centers/scales the log-target internally for a well-conditioned fit.
    gpr = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                   n_restarts_optimizer=2, random_state=42)
    model = TransformedTargetRegressor(
        regressor=Pipeline([('scaler', StandardScaler()), ('gpr', gpr)]),
        func=np.log1p, inverse_func=np.expm1,
    )
    model.fit(train_x, y_train)

    # Predict mean and 1-sigma uncertainty (in log space), then back-transform.
    inner = model.regressor_
    tr_scaled = inner.named_steps['scaler'].transform(train_x)
    va_scaled = inner.named_steps['scaler'].transform(valid_x)
    tr_mu_log = inner.named_steps['gpr'].predict(tr_scaled)
    va_mu_log, va_sd_log = inner.named_steps['gpr'].predict(va_scaled, return_std=True)
    train_pred = np.expm1(tr_mu_log)
    pred = np.expm1(va_mu_log)

    results.append({
        'target': target,
        'train_r2': r2_score(y_train, train_pred),
        'valid_r2': r2_score(y_valid, pred),
        'train_r2_log': r2_score(np.log1p(y_train), tr_mu_log),
        'valid_r2_log': r2_score(np.log1p(y_valid), va_mu_log),
        'MAE': mean_absolute_error(y_valid, pred),
        'RMSE': np.sqrt(mean_squared_error(y_valid, pred)),
        'mean_1sigma_log': va_sd_log.mean(),   # avg predictive uncertainty (log units)
    })

results_df = pd.DataFrame(results).set_index('target')
print(results_df.round(4).to_string())


c:\Users\avyay\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


                  train_r2  valid_r2  train_r2_log  valid_r2_log        MAE       RMSE  mean_1sigma_log
target                                                                                                 
avg_length_um       0.9969    0.7582        0.9966        0.8888   133.4152   389.5291           0.5658
avg_volume_um3      0.9986    0.0016        0.9980        0.8689  1772.5623  9010.5653           0.7365
total_tips          0.9435    0.7192        0.9608        0.8670    34.4238   113.2195           0.5488
n_apical_events     0.7966    0.5528        0.9149        0.8288    19.4690    66.5819           0.7657
n_lateral_events    1.0000    0.6559        1.0000        0.9263    22.6776    78.3581           0.3926
